# 3 — Advanced Examples

This notebook demonstrates the full capabilities of the package:

1. Solving a 1-D PDE (heat equation)
2. Bayesian optimisation of reservoir parameters
3. Sensitivity analysis
4. Designing a custom reaction network
5. Multi-horizon comparison
6. Chaotic system (Lorenz attractor)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import chemical_reservoir as cr
from pathlib import Path

NETWORK_DIR = Path("../network_configs")

---
## 3.1  Solving a PDE — Heat Equation

The reservoir can solve 1-D PDEs via the method of lines.
The PDE $\partial_t u = \alpha\, \partial_{xx} u$ is semi-discretised
in space and treated as a system of coupled ODEs.

In [ ]:
# Create a heat equation problem
heat = cr.problems.create_heat_equation_problem(t_end=2.0, alpha=0.1)
print(f"PDE: {heat.name}")
print(f"Spatial domain: {heat.x_domain}")
print(f"Time span: {heat.t_span}")
print(f"Spatial grid points: {heat.n_spatial}")

In [ ]:
# Load reservoir and solve the PDE
reservoir, config = cr.io.load_reservoir_from_json(
    NETWORK_DIR / "cascade.json"
)

pde_results = reservoir.solve_pde(
    heat,
    n_train_time=80,
    n_test_time=40,
    test_horizon_factor=0.3,
)

print("PDE result keys:", list(pde_results.keys()))

In [ ]:
# Evaluate PDE metrics
pde_metrics = cr.MetricsCalculator.calculate_pde_metrics(
    sol_true=pde_results['sol_true'],
    sol_pred=pde_results['sol_pred'],
    x=pde_results['x_grid'],
    t=pde_results['t_all'],
    n_train=len(pde_results['t_train']),
)

print(f"Train RMSE: {pde_metrics['train_rmse']:.4e}")
print(f"Test  RMSE: {pde_metrics['test_rmse']:.4e}")

In [ ]:
# Visualise PDE solution
n_train = len(pde_results['t_train'])
x = pde_results['x_grid']
t = pde_results['t_all']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

im0 = axes[0].pcolormesh(x, t, pde_results['sol_true'], shading='auto', cmap='viridis')
axes[0].set(xlabel='x', ylabel='t', title='True Solution')
axes[0].axhline(t[n_train - 1], color='w', ls='--', lw=1)
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(x, t, pde_results['sol_pred'], shading='auto', cmap='viridis')
axes[1].set(xlabel='x', ylabel='t', title='Predicted Solution')
axes[1].axhline(t[n_train - 1], color='w', ls='--', lw=1)
plt.colorbar(im1, ax=axes[1])

error = np.abs(pde_results['sol_true'] - pde_results['sol_pred'])
im2 = axes[2].pcolormesh(x, t, error, shading='auto', cmap='hot')
axes[2].set(xlabel='x', ylabel='t', title='Absolute Error')
axes[2].axhline(t[n_train - 1], color='w', ls='--', lw=1)
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

---
## 3.2  Bayesian Optimisation

Optimise input rates and initial concentrations using a Gaussian Process
surrogate and Expected Improvement acquisition.

In [ ]:
# Create a fresh reservoir for optimisation
reservoir_opt, _ = cr.io.load_reservoir_from_json(
    NETWORK_DIR / "simple_autocatalytic.json"
)

problem = cr.problems.create_vanderpol_problem()

# Run Bayesian optimisation (reduce iterations for demo)
opt_results = reservoir_opt.optimize_with_bayesian(
    problem,
    n_iterations=20,
    n_initial=8,
    verbose=True,
)

In [ ]:
print(f"Best RMSE: {opt_results.get('best_rmse', 'N/A')}")

# Visualise optimisation convergence
vis = cr.ResultsVisualizer()
vis.plot_optimization_analysis(opt_results)

---
## 3.3  Sensitivity Analysis

Assess robustness to perturbations in:
- Species concentrations
- Input rates
- Measurement channel selection
- Local parameter variations

In [ ]:
reservoir_sa, _ = cr.io.load_reservoir_from_json(
    NETWORK_DIR / "cascade.json"
)
problem = cr.problems.create_schnakenberg_problem()

analyzer = cr.SensitivityAnalyzer(reservoir_sa, problem)

print("Running concentration sensitivity...")
analyzer.analyze_concentration_sensitivity(n_train=200)

print("Running input rate sensitivity...")
analyzer.analyze_input_rate_sensitivity(n_train=200)

print("Running measurement sensitivity...")
analyzer.analyze_measurement_sensitivity(n_train=200)

print("Running local parameter sensitivity...")
analyzer.compute_local_sensitivity(n_train=200)

print("Done!")

In [ ]:
# Visualise sensitivity results
vis = cr.ResultsVisualizer()
vis.plot_sensitivity_analysis(analyzer.sensitivity_results)

---
## 3.4  Designing a Custom Reaction Network

You can define your own network and save it as JSON.

In [ ]:
# Define a custom 4-species network with inhibition
custom_reactions = [
    {"species": ["X", "Y"], "reactants": [1, 0], "products": [0, 1], "rate": 1.2, "name": "activation"},
    {"species": ["Y", "Z"], "reactants": [1, 0], "products": [0, 1], "rate": 0.8, "name": "cascade"},
    {"species": ["Z", "W"], "reactants": [1, 0], "products": [0, 1], "rate": 0.6, "name": "production"},
    {"species": ["W", "X"], "reactants": [1, 1], "products": [0, 0], "rate": 0.3, "name": "inhibition"},
    # Reverse / degradation
    {"species": ["Y", "X"], "reactants": [1, 0], "products": [0, 1], "rate": 0.1, "name": "rev1"},
    {"species": ["Z", "Y"], "reactants": [1, 0], "products": [0, 1], "rate": 0.08, "name": "rev2"},
]

# Save to JSON
cr.io.save_reaction_network(
    filepath="custom_inhibition.json",
    reactions=custom_reactions,
    input_channels=["X", "Y"],
    output_channels=["X", "Y", "Z", "W"],
    name="custom_inhibition",
    description="4-species network with negative feedback via inhibition",
)

# Load it back and use it
res_custom, cfg_custom = cr.io.load_reservoir_from_json("custom_inhibition.json")
print(f"Custom network: {cfg_custom['name']}, {len(cfg_custom['reactions'])} reactions")

In [ ]:
# Test custom network on Van der Pol
problem = cr.problems.create_vanderpol_problem()
results_custom = res_custom.solve_ode(problem, n_train=400, n_test=200)

met_custom = cr.MetricsCalculator.calculate_all(
    y_true=results_custom['y_true'],
    y_pred=results_custom['y_pred'],
    t=results_custom['t_all'],
    n_train=len(results_custom['t_train']),
    variable_names=problem.variable_names,
)
print(f"Custom network — Test RMSE: {met_custom.test_rmse:.4e}, R²: {met_custom.test_r2:.4f}")

---
## 3.5  Multi-Horizon Comparison

Evaluate how prediction accuracy degrades as the test horizon increases.

In [ ]:
reservoir_hz, _ = cr.io.load_reservoir_from_json(NETWORK_DIR / "cascade.json")
problem = cr.problems.create_vanderpol_problem()

horizons = [0.1, 0.25, 0.5, 0.75, 1.0, 1.5]
rmse_results = []
r2_results = []

for h in horizons:
    res = reservoir_hz.solve_ode(problem, n_train=400, n_test=200,
                                 test_horizon_factor=h)
    met = cr.MetricsCalculator.calculate_all(
        y_true=res['y_true'], y_pred=res['y_pred'],
        t=res['t_all'], n_train=len(res['t_train']),
        variable_names=problem.variable_names,
    )
    rmse_results.append(met.test_rmse)
    r2_results.append(met.test_r2)
    print(f"Horizon {h:.2f}: RMSE={met.test_rmse:.4e}, R²={met.test_r2:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.semilogy(horizons, rmse_results, 'bo-', lw=2)
ax1.set(xlabel='Test Horizon Factor', ylabel='Test RMSE', title='Error vs Horizon')
ax1.grid(True, alpha=0.3)

ax2.plot(horizons, r2_results, 'rs-', lw=2)
ax2.set(xlabel='Test Horizon Factor', ylabel='Test R²', title='Accuracy vs Horizon')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3.6  Chaotic System — Lorenz Attractor

A challenging test: can the reservoir track a chaotic trajectory?

> **Note**: The Lorenz system is chaotic — prediction errors grow exponentially
> with a timescale of ~1.1 time units.  Open-loop prediction (used below)
> diverges rapidly.  Switching to **closed-loop mode** (`solve_ode_closed_loop`)
> significantly extends the useful prediction horizon.  See **Notebook 05** for
> a detailed comparison.

In [ ]:
lorenz = cr.problems.create_lorenz_problem(t_end=10.0)

reservoir_lorenz, _ = cr.io.load_reservoir_from_json(NETWORK_DIR / "dense.json")
res_lorenz = reservoir_lorenz.solve_ode(lorenz, n_train=600, n_test=200)

met_lorenz = cr.MetricsCalculator.calculate_all(
    y_true=res_lorenz['y_true'], y_pred=res_lorenz['y_pred'],
    t=res_lorenz['t_all'], n_train=len(res_lorenz['t_train']),
    variable_names=lorenz.variable_names,
)

print(f"Lorenz — Test RMSE: {met_lorenz.test_rmse:.4e}, R²: {met_lorenz.test_r2:.4f}")

In [ ]:
# 3D phase portrait
from mpl_toolkits.mplot3d import Axes3D

n_train = len(res_lorenz['t_train'])
y_true = res_lorenz['y_true']
y_pred = res_lorenz['y_pred']

fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.plot(y_true[:n_train, 0], y_true[:n_train, 1], y_true[:n_train, 2],
         'b-', alpha=0.5, label='Train (true)')
ax1.plot(y_true[n_train:, 0], y_true[n_train:, 1], y_true[n_train:, 2],
         'g-', alpha=0.8, label='Test (true)')
ax1.set_title('True Lorenz')
ax1.legend(fontsize=8)

ax2 = fig.add_subplot(122, projection='3d')
ax2.plot(y_pred[:n_train, 0], y_pred[:n_train, 1], y_pred[:n_train, 2],
         'b-', alpha=0.5, label='Train (pred)')
ax2.plot(y_pred[n_train:, 0], y_pred[n_train:, 1], y_pred[n_train:, 2],
         'r-', alpha=0.8, label='Test (pred)')
ax2.set_title('Reservoir Prediction')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 3.7  Other PDE Problems

In [ ]:
# Fisher equation — reaction-diffusion with travelling front
fisher = cr.problems.create_fisher_equation_problem()

reservoir_fisher, _ = cr.io.load_reservoir_from_json(NETWORK_DIR / "cascade.json")
res_fisher = reservoir_fisher.solve_pde(fisher, n_train_time=80, n_test_time=40)

met_fisher = cr.MetricsCalculator.calculate_pde_metrics(
    sol_true=res_fisher['sol_true'], sol_pred=res_fisher['sol_pred'],
    x=res_fisher['x_grid'], t=res_fisher['t_all'],
    n_train=len(res_fisher['t_train']),
)
print(f"Fisher PDE — Test RMSE: {met_fisher['test_rmse']:.4e}")

In [ ]:
# Built-in PDE dashboard
vis = cr.ResultsVisualizer()
vis.plot_pde_metrics(res_fisher, met_fisher)

---
## Summary

This notebook demonstrated:

- **PDE solving** via method of lines + reservoir readout
- **Bayesian optimisation** for automatic hyperparameter tuning
- **Sensitivity analysis** for robustness evaluation
- **Custom network design** with JSON I/O
- **Multi-horizon** degradation studies
- **Chaotic systems** (Lorenz attractor)

See **Notebook 04** for a detailed API reference, and **Notebook 05** for a
side-by-side comparison of open-loop vs closed-loop prediction modes.